# Generating Video Highlights
> https://pyimagesearch.com/2025/06/30/generating-video-highlights-using-the-smolvlm2-model/

## Setup and Imports

In [1]:
import os
import json
import torch
import tempfile
import gradio as gr
import logging
import subprocess
from pathlib import Path
from transformers import AutoProcessor, AutoModelForImageTextToText

## Setup Logger

In [2]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## Get Video Duration in Seconds

In [3]:
def get_video_duration_seconds(video_path: str) -> float:
   """Use ffprobe to get video duration in seconds."""
   cmd = [
       "ffprobe",
       "-v", "quiet",
       "-print_format", "json",
       "-show_format",
       video_path
   ]
   result = subprocess.run(cmd, capture_output=True, text=True)
   info = json.loads(result.stdout)
   return float(info["format"]["duration"])

## Load Model and Processor

In [5]:
def load_model_and_processor(model_path: str, device: str = "mps", dtype=torch.bfloat16):
   processor = AutoProcessor.from_pretrained(model_path)
   model = AutoModelForImageTextToText.from_pretrained(
       model_path,
       torch_dtype=dtype,
       _attn_implementation="flash_attention_2"
   ).to(device)
   return processor, model

## Analyze Video Content

In [8]:
def analyze_video_content(processor, model, video_path: str, device: str = "mps") -> str:
   system_message = "You are a helpful assistant that can understand videos. Describe what type of video this is and what's happening in it."
   messages = [
       {
           "role": "system",
           "content": [{"type": "text", "text": system_message}]
       },
       {
           "role": "user",
           "content": [
               {"type": "video", "path": video_path},
               {"type": "text", "text": "What type of video is this and what's happening in it? Be specific about the content type and general activities you observe."}
           ]
       }
   ]

   inputs = processor.apply_chat_template(
       messages,
       add_generation_prompt=True,
       tokenize=True,
       return_dict=True,
       return_tensors="pt"
   ).to(device, dtype=torch.bfloat16)
   outputs = model.generate(**inputs, max_new_tokens=512, do_sample=True, temperature=0.7)
   return processor.decode(outputs[0], skip_special_tokens=True).lower().split("assistant: ")[1]

## Determine Highlights

In [14]:
def determine_highlights(processor, model, video_description: str, prompt_num: int = 1, device: str = "mps") -> str:
    system_prompts = {
        1: "You are a highlight editor. List archetypal dramatic moments that would make compelling highlights if they appear in the video. Each moment should be specific enough to be recognizable but generic enough to potentially exist in other videos of this type.",
        2: "You are a helpful visual-language assistant that can understand videos and edit. You are tasked with helping the user to create highlight reels for videos. Highlights should be rare and important events in the video in question."
    }
    user_prompts = {
        1: "List potential highlight moments to look for in this video:",
        2: "List dramatic moments that would make compelling highlights if they appear in the video. Each moment should be specific enough to be recognizable but generic enough to potentially exist in any video of this type:"
    }

    messages = [
        {
            "role": "system",
            "content": [{"type": "text", "text": system_prompts[prompt_num]}]
        },
        {
            "role": "user",
            "content": [{"type": "text", "text": f"""Here is a description of a video:\n\n{video_description}\n\n{user_prompts[prompt_num]}"""}]
        }
    ]
    print(f"Using prompt {prompt_num} for highlight detection")
    print(messages)

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(device, dtype=torch.bfloat16)
    
    outputs = model.generate(**inputs, max_new_tokens=256, do_sample=True, temperature=0.7)
    return processor.decode(outputs[0], skip_special_tokens=True).split("Assistant: ")[1]

## Process Video Segment

In [15]:
def process_segment(processor, model, video_path: str, highlight_types: str, device: str = "mps") -> bool:
    messages = [
        {
            "role": "system",
            "content": [{"type": "text", "text": "You are a video highlight analyzer. Your role is to identify moments that have high dramatic value, focusing on displays of skill, emotion, personality, or tension. Compare video segments against provided example highlights to find moments with similar emotional impact and visual interest, even if the specific actions differ."}]
        },
        {
            "role": "user",
            "content": [
                {"type": "video", "path": video_path},
                {"type": "text", "text": f"""Given these highlight examples:\n{highlight_types}\n\nDoes this video contain a moment that matches the core action of one of the highlights? Answer with:\n'yes' or 'no'\nIf yes, justify it"""}
            ]
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(device, dtype=torch.bfloat16)
   
    outputs = model.generate(**inputs, max_new_tokens=64, do_sample=False)
    response = processor.decode(outputs[0], skip_special_tokens=True).lower().split("assistant: ")[1]
    print(f"Segment response {response}")
    return "yes" in response